This notebook is dedicated to performing **Exploratory Data Analysis (EDA)** on the enriched Superstore dataset (`superstore_features.pkl`). 

The primary goal here is to answer core business questions through statistical aggregations, summary metrics, and group breakdowns, establishing the exact data points that will drive our visualizations in Notebook 04.

---

## Workflow Steps
1. **Setup & Data Ingestion:** Load processed data with engineered features intact.
2. **Univariate Analysis:** Inspect distributions, summary statistics (mean, median, quantiles), and value counts.
3. **Bivariate & Multivariate Analysis:** Explore relationships (e.g., Discount vs. Profitability, Shipping Duration vs. Delays).
4. **Answering Core Business Questions:** Produce aggregate tables for financial performance, product categories, and customer metrics.
5. **Insights Synthesis:** Document key data-driven findings to inform visual storytelling.

---

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

data_path = "../data/processed/superstore_features.pkl"
df = pd.read_pickle(data_path)

float_cols = df.select_dtypes(include=['float32', 'float64']).columns
df[float_cols] = df[float_cols].astype('float64').round(2)

print(f"Data Shape: {df.shape}")

df.head(3)

Data Shape: (9994, 35)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping_Duration,Is_Delayed,Order_Year,Order_Month,Order_Month_Name,Order_Day_Name,Order_Quarter,Is_Weekend,Profit_Margin,Discount_Amount,Unit_Price,Customer_Order_Count,Customer_Total_Spend,Order_Size
0,1,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,3,0,2018,11,November,Thursday,2018Q4,0,0.16,0.00,130.98,3,1148.78,Small
1,2,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,3,0,2018,11,November,Thursday,2018Q4,0,0.30,0.00,243.98,3,1148.78,Medium
2,3,CA-2018-138688,2018-06-12,2018-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,4,0,2018,6,June,Tuesday,2018Q2,0,0.47,0.00,7.31,5,1119.48,Small


In [2]:
num_cols = [
    'Sales', 'Profit', 'Discount', 'Shipping_Duration', 
    'Profit_Margin', 'Discount_Amount', 'Unit_Price', 
    'Customer_Total_Spend', 'Customer_Order_Count'
]

eda_summary = df[num_cols].describe().T[['mean', '50%', 'min', 'max', 'std']]
eda_summary.rename(columns={'50%': 'median'}, inplace=True)
eda_summary = eda_summary.round(2)
print("Summary Statistics")
eda_summary

Summary Statistics


,mean,median,min,max,std
Sales,229.86,54.49,0.44,22638.48,623.25
Profit,28.66,8.66,-6599.98,8399.98,234.26
Discount,0.16,0.20,0.00,0.80,0.21
Shipping_Duration,3.96,4.00,0.00,7.00,1.75
Profit_Margin,0.12,0.27,-2.75,0.50,0.47
Discount_Amount,32.28,1.04,0.00,11319.24,164.03
Unit_Price,60.92,16.27,0.34,3773.08,142.93
Customer_Total_Spend,3593.69,2874.34,4.83,25043.05,2837.91
Customer_Order_Count,7.36,7.00,1.00,17.00,2.55


In [3]:
#  Financial Performance Breakdown: Category & Sub-Category

category_performance = df.groupby(['Category', 'Sub-Category'], observed=True).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean'),
    Avg_Profit_Margin=('Profit_Margin', 'mean'),
    Total_Orders=('Order ID', 'nunique')
).reset_index()

category_performance = category_performance.round(2)


category_performance = category_performance.sort_values(by='Total_Profit', ascending=False)

print("Financial Breakdown by Product Category & Sub-Category:")
category_performance

Financial Breakdown by Product Category & Sub-Category:


,Category,Sub-Category,Total_Sales,Total_Profit,Avg_Discount,Avg_Profit_Margin,Total_Orders
14,Technology,Copiers,149528.01,55617.85,0.16,0.32,68
16,Technology,Phones,330007.10,44516.04,0.15,0.12,814
13,Technology,Accessories,167380.31,41936.73,0.08,0.22,718
10,Office Supplies,Paper,78479.24,34053.16,0.07,0.43,1191
6,Office Supplies,Binders,203412.71,30221.42,0.37,-0.20,1316
1,Furniture,Chairs,328449.08,26590.11,0.17,0.04,576
11,Office Supplies,Storage,223843.59,21278.96,0.07,0.09,777
4,Office Supplies,Appliances,107532.14,18138.00,0.17,-0.16,451
2,Furniture,Furnishings,91705.12,13059.19,0.14,0.14,877
7,Office Supplies,Envelopes,16476.38,6964.03,0.08,0.42,249


In [4]:
# Discount Sensitivity & Profitability Analysis

discount_analysis = df.groupby('Discount', observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Profit_Margin=('Profit_Margin', 'mean'),
    Total_Items=('Order ID', 'count'),
    Loss_Making_Orders=('Profit', lambda x: (x < 0).sum())
).reset_index()

discount_analysis['Loss_Rate_%'] = (
    discount_analysis['Loss_Making_Orders'] / discount_analysis['Total_Items'] * 100
)

discount_analysis = discount_analysis.round(2)

print("Profitability and Loss Breakdown Across Discount Rates:")
discount_analysis

Profitability and Loss Breakdown Across Discount Rates:


,Discount,Total_Sales,Total_Profit,Avg_Profit_Margin,Total_Items,Loss_Making_Orders,Loss_Rate_%
0,0.00,1087908.47,320987.10,0.34,4798,0,0.00
1,0.10,54369.30,9029.20,0.16,94,4,4.26
2,0.15,27558.55,1418.97,0.03,52,17,32.69
3,0.20,764594.28,90337.58,0.18,3657,502,13.73
4,0.30,103226.67,-10369.34,-0.11,227,208,91.63
5,0.32,14493.45,-2391.16,-0.17,27,27,100.00
6,0.40,116417.83,-23057.07,-0.22,206,180,87.38
7,0.45,5484.98,-2493.12,-0.46,11,11,100.00
8,0.50,58918.56,-20506.48,-0.55,66,66,100.00
9,0.60,6644.68,-5944.62,-0.69,138,138,100.00


In [5]:
# Feature Prep: Compute Actual Delay Days (SLA-based)

sla_thresholds = {
    'Same Day': 0,
    'First Class': 2,
    'Second Class': 3,
    'Standard Class': 5
}

# Calculate expected threshold per shipment class
df['Expected_Max_Days'] = df['Ship Mode'].map(sla_thresholds).astype('float64')

# Calculate delay days (clip at 0 so on-time shipments stay 0)
df['Delay_Days'] = (df['Shipping_Duration'] - df['Expected_Max_Days']).clip(lower=0)


df[['Ship Mode', 'Shipping_Duration', 'Expected_Max_Days', 'Delay_Days']].head(3)

,Ship Mode,Shipping_Duration,Expected_Max_Days,Delay_Days
0,Second Class,3,3.00,0.00
1,Second Class,3,3.00,0.00
2,Second Class,4,3.00,1.00


In [6]:
# Logistics & Shipping Performance Analysis

shipping_analysis = df.groupby('Ship Mode', observed=False).agg(
    Total_Orders=('Order ID', 'nunique'),
    Avg_Shipping_Days=('Shipping_Duration', 'mean'),
    Min_Shipping_Days=('Shipping_Duration', 'min'),
    Max_Shipping_Days=('Shipping_Duration', 'max'),
    Delayed_Orders=('Is_Delayed', 'sum'),
    Avg_Delay_Days=('Delay_Days', 'mean'),
    Max_Delay_Days=('Delay_Days', 'max')
).reset_index()


shipping_analysis['Delay_Rate_%'] = (
    shipping_analysis['Delayed_Orders'] / shipping_analysis['Total_Orders'] * 100
)


shipping_analysis = shipping_analysis.round(2)


shipping_analysis = shipping_analysis.sort_values(by='Total_Orders', ascending=False)

print("Shipping Performance & Delivery Delay Summary:")
shipping_analysis

Shipping Performance & Delivery Delay Summary:


,Ship Mode,Total_Orders,Avg_Shipping_Days,Min_Shipping_Days,Max_Shipping_Days,Delayed_Orders,Avg_Delay_Days,Max_Delay_Days,Delay_Rate_%
3,Standard Class,2994,5.01,3,7,3564,0.41,2.00,119.04
2,Second Class,964,3.24,1,5,429,0.63,2.00,44.50
0,First Class,787,2.18,1,5,1,0.41,3.00,0.13
1,Same Day,264,0.04,0,1,0,0.04,1.00,0.00


In [7]:
# Customer Segment & Top VIP Customers Breakdown
# 1. Performance breakdown by Customer Segment
segment_summary = df.groupby('Segment', observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Unique_Customers=('Customer ID', 'nunique'),
    Total_Orders=('Order ID', 'nunique'),
    Avg_Profit_Margin=('Profit_Margin', 'mean')
).reset_index().round(2)

print("Segment Summary:")
display(segment_summary)

# 2. Extract Top 10 High-Value VIP Customers by Total Spend
top_vip_customers = df.groupby(['Customer ID', 'Customer Name', 'Segment'], observed=False).agg(
    Total_Spend=('Sales', 'sum'),
    Total_Profit_Generated=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique'),
    Avg_Discount_Taken=('Discount', 'mean')
).reset_index().sort_values(by='Total_Spend', ascending=False).head(10).round(2)

print("\n Top 10 VIP Customers:")
top_vip_customers

Segment Summary:


,Segment,Total_Sales,Total_Profit,Unique_Customers,Total_Orders,Avg_Profit_Margin
0,Consumer,1161401.21,134118.70,409,2586,0.11
1,Corporate,706146.33,91979.10,236,1514,0.12
2,Home Office,429653.20,60298.75,148,909,0.14



 Top 10 VIP Customers:


,Customer ID,Customer Name,Segment,Total_Spend,Total_Profit_Generated,Total_Orders,Avg_Discount_Taken
1667360,SM-20320,Sean Miller,Home Office,25043.07,-1980.75,5,0.25
1765030,TC-20980,Tamara Chand,Corporate,19052.22,8981.32,5,0.12
1479225,RB-19360,Raymond Buch,Consumer,15117.35,6976.09,6,0.09
1738943,TA-21385,Tom Ashbrook,Home Office,14595.62,4703.79,4,0.08
14292,AB-10105,Adrian Barton,Consumer,14473.57,5444.81,10,0.24
1033809,KL-16645,Ken Lonsdale,Consumer,14175.23,806.84,12,0.20
1593564,SC-20095,Sanjit Chand,Consumer,14142.34,5757.42,9,0.06
778935,HL-15040,Hunter Lopez,Consumer,12873.30,5622.43,6,0.02
1626873,SE-20110,Sanjit Engle,Consumer,12209.44,2650.68,11,0.11
312117,CC-12370,Christopher Conant,Consumer,12129.08,2177.05,5,0.28


In [8]:
# Time Series & Seasonality Performance Analysis

yearly_trend = df.groupby('Order_Year', observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique'),
    Avg_Profit_Margin=('Profit_Margin', 'mean')
).reset_index().round(2)

print("Yearly Performance Summary:")
display(yearly_trend)

monthly_seasonality = df.groupby(['Order_Month', 'Order_Month_Name'], observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique')
).reset_index().sort_values(by='Order_Month').round(2)

print("\n Monthly Seasonality Breakdown:")
monthly_seasonality

Yearly Performance Summary:


,Order_Year,Total_Sales,Total_Profit,Total_Orders,Avg_Profit_Margin
0,2016,484247.48,49543.87,969,0.12
1,2017,470532.42,61618.43,1038,0.12
2,2018,609205.74,81794.94,1315,0.13
3,2019,733215.10,93439.31,1687,0.12



 Monthly Seasonality Breakdown:


,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Total_Orders
0,1,January,94924.87,9134.46,178
1,2,February,59751.26,10294.59,162
2,3,March,205005.51,28594.60,354
3,4,April,137762.15,11587.48,343
4,5,May,155028.80,22411.25,369
5,6,June,152718.65,21285.77,364
6,7,July,147238.11,13832.62,338
7,8,August,159043.98,21777.01,341
8,9,September,307649.91,36857.29,688
9,10,October,200322.97,31784.07,417


In [9]:
# 8. Regional Performance Breakdown & Final Data Export

regional_summary = df.groupby('Region', observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique'),
    Avg_Discount=('Discount', 'mean'),
    Avg_Profit_Margin=('Profit_Margin', 'mean')
).reset_index().sort_values(by='Total_Profit', ascending=False).round(2)

print("Regional Performance:")
display(regional_summary)

Regional Performance:


,Region,Total_Sales,Total_Profit,Total_Orders,Avg_Discount,Avg_Profit_Margin
3,West,725457.84,108418.32,1611,0.11,0.22
1,East,678781.27,91522.50,1401,0.15,0.17
2,South,391721.86,46749.46,822,0.15,0.16
0,Central,501239.77,39706.27,1175,0.24,-0.10


In [10]:
output_path = "../data/processed/superstore_eda_ready.pkl"
df.to_pickle(output_path)